In [13]:
import json
import pandas as pd

with open("data/bands_genres_cleaned.json", encoding="utf-8") as f:
    bands = json.load(f)

bands = pd.DataFrame(list(bands.items()), columns=["artiste", "genre"])
print(bands["genre"].apply(type).value_counts())
print(bands.head())

genre
<class 'list'>        72
<class 'NoneType'>    28
Name: count, dtype: int64
        artiste                                    genre
0  Judas Priest                            [Heavy Metal]
1  Alice Cooper                                     None
2          Rage              [Heavy, Speed, Power Metal]
3  Unholy Grave      [Grindcore, Death Metal, Noisecore]
4  Napalm Death  [Hardcore Punk, Grindcore, Death Metal]


In [14]:
with open("data/dataset.json", encoding="utf-8") as f:
    chansons = json.load(f)

rows=[]
for _,artist in chansons["dataset"].items() :
    for _, album in artist.get("albums", {}).items():
        for song in album.get("songs", []):
            rows.append({
                "artiste" : artist.get("name"),
                "album" : album.get("name"),
                "annee" : album.get("release_year"),
                "titre" : song.get("title"),
                "paroles" : song.get("lyrics")
            })

songs = pd.DataFrame(rows)
print(songs.shape)
print(songs.head())

(257716, 5)
  artiste                 album annee      titre  \
0     '68  In Humor And Sadness  2014  Track 1 R   
1     '68  In Humor And Sadness  2014  Track 2 E   
2     '68  In Humor And Sadness  2014  Track 3 G   
3     '68  In Humor And Sadness  2014  Track 4 R   
4     '68  In Humor And Sadness  2014  Track 5 E   

                                             paroles  
0  Take your heart into the next room.\nMake a sc...  
1  It doesn't matter what you say,\nThey could al...  
2  Hey kid, don't listen to them.\nWe got a long,...  
3  Stained glass.\nI wish that I could say that i...  
4  I've been taken to the edge of it all.\nI've b...  


In [20]:
bands["genre"] = bands["genre"].apply(lambda l: [g.replace("Metal", "").strip() for g in l] if isinstance(l, list) else l)
genres= []
for i in range(len(bands["genre"])):
    if bands["genre"][i] is not None :
        for j in range(len(bands["genre"][i])) :
            if bands["genre"][i][j] not in genres :
                genres.append(bands["genre"][i][j])
len(genres)

43

In [21]:
chansons = []
for g in genres:
    masque = bands["genre"].apply(lambda l: isinstance(l, list) and g in l) # test si genre dans la liste des genres de l'artiste
    positions = bands.loc[masque].set_index("artiste")["genre"].apply(lambda l: l.index(g)) # garde la position du genre si artiste a pls genres
    genre_songs = songs[songs["artiste"].isin(positions.index)].copy()
    genre_songs["genre"] = g
    genre_songs["position"] = genre_songs["artiste"].map(positions)
    chansons.append(genre_songs)

songs_genre = (pd.concat(chansons, ignore_index=True).sort_values(["genre", "artiste", "titre"]).reset_index(drop=True))

songs_genre.to_csv("all_songs.csv")

In [3]:
songs[songs["artiste"] == "'68"]

,artiste,album,annee,titre,paroles
0,'68,In Humor And Sadness,2014,Track 1 R,Take your heart into the next room.\nMake a sc...
1,'68,In Humor And Sadness,2014,Track 2 E,"It doesn't matter what you say,\nThey could al..."
2,'68,In Humor And Sadness,2014,Track 3 G,"Hey kid, don't listen to them.\nWe got a long,..."
3,'68,In Humor And Sadness,2014,Track 4 R,Stained glass.\nI wish that I could say that i...
4,'68,In Humor And Sadness,2014,Track 5 E,I've been taken to the edge of it all.\nI've b...
5,'68,In Humor And Sadness,2014,Track 6 T,Over thought.\nUndersold.\nI have sung every s...
6,'68,In Humor And Sadness,2014,Track 7 N,"The future has seen better days.\nAlways, clos..."
7,'68,In Humor And Sadness,2014,Track 8 O,"Hey, I still got a lot to say.\nMaybe I'm wron..."
8,'68,In Humor And Sadness,2014,Track 9 T,I can understand your hesitation but\nYou can'...
9,'68,In Humor And Sadness,2014,Track 10 .,Barricade all of your fears and we'll speak to...


In [4]:
songs.groupby("artiste")["album"].nunique() 

artiste
'68                1
(Sic)Monic         2
+\-                1
...Aaaarrghh...    3
...And Oceans      5
                  ..
Ásmegin            2
Íon                2
Öoooooooooo        1
Øscillatör         1
Žalvarinis         1
Name: album, Length: 8637, dtype: int64

In [5]:
songs.query("artiste == '(Sic)Monic' and album == 'Look To The Skies'")

,artiste,album,annee,titre,paroles
10,(Sic)Monic,Look To The Skies,2006,Fist To Throat,Every time I look I feel\nevery time sentiment...
11,(Sic)Monic,Look To The Skies,2006,Seven Inches Deep,I heard a man once say before he faded away I ...
12,(Sic)Monic,Look To The Skies,2006,My Own Reflection,Ill be the victim bleeding in the gutter you b...
13,(Sic)Monic,Look To The Skies,2006,Blood Shot,This is the blood of the weak blanketed in sac...
14,(Sic)Monic,Look To The Skies,2006,Silent Screams,[Pre-verse:]\n\nInfinite is how I dream silent...
15,(Sic)Monic,Look To The Skies,2006,Hypnotic,So beautiful your own sense of hypnotic face d...
16,(Sic)Monic,Look To The Skies,2006,Shadow Bound,Never meant to compromise born to indefinitely...
17,(Sic)Monic,Look To The Skies,2006,Forever And A Day,[Chorus:]\n\nWhence forth she came we breathed...
18,(Sic)Monic,Look To The Skies,2006,Dementia,[Chorus:]\n\nCatatonic so symphonic eyes demon...
19,(Sic)Monic,Look To The Skies,2006,Desensitized,Suffocate the voices penetrate the noises thro...


In [6]:
bands_long = bands.explode("genre").reset_index(drop=True) #une ligne par genre
# on retire le mot Metal des genres
bands_long["genre"] = (
    bands_long["genre"]
    .str.replace(r"\bmetal\b", "", case=False, regex=True)
    .str.strip()
)
heavy = bands_long.query("genre == 'Heavy'")["artiste"]
heavy

0        Judas Priest
2                Rage
12          Motorhead
22              Saxon
34       Grave Digger
36           W.A.S.P.
41     Axel Rudi Pell
46              Anvil
51           Loudness
56         Mägo De Oz
61           Savatage
65      Black Sabbath
67      Virgin Steele
69      Virgin Steele
85           Megadeth
89       Running Wild
92             Accept
106      King Diamond
115         Metallica
129     Ozzy Osbourne
131           Stryper
144       Iron Maiden
167           Manowar
169             Lordi
182               Dio
183            Dokken
Name: artiste, dtype: object

In [7]:
songs.query("artiste in @heavy").head()

,artiste,album,annee,titre,paroles
4203,Accept,Accept,1979,Lady Lou,Running through empty streets\nTry to find my ...
4204,Accept,Accept,1979,Tired Of Me,Rain wets your body - rain wets the streets\nB...
4205,Accept,Accept,1979,Seawinds,In a cold and dark December\nAs I walked into ...
4206,Accept,Accept,1979,Take Him In My Heart,Once upon a time\nThere was a girl so young an...
4207,Accept,Accept,1979,Sounds Of War,Sounds of chains we hear from far behind\nMech...


In [8]:
doom = bands_long.query("genre == 'Doom'")["artiste"]

In [9]:
doom_songs = songs.query("artiste in @doom").copy()
doom_songs.head()

,artiste,album,annee,titre,paroles
12648,Amorphis,The Karelian Isthmus,1992,Karelia,[lyrics: E. Holopainen]\n\n [music: T. Koivusa...
12649,Amorphis,The Karelian Isthmus,1992,The Gathering,[lyrics: E. Holopainen]\n\n [music: T. Koivusa...
12650,Amorphis,The Karelian Isthmus,1992,Grails Mysteries,[lyrics &amp; music: E. Holopainen]\n\nI shall...
12651,Amorphis,The Karelian Isthmus,1992,Warriors Trail,[lyrics: E. Holopainen]\n\n [music: T. Koivusa...
12652,Amorphis,The Karelian Isthmus,1992,Black Embrace,[lyrics: J. Rechberger]\n\n [music: T. Koivusa...


In [10]:
album_Karelian = doom_songs.query("album == 'The Karelian Isthmus'")[["titre","paroles"]]
album_Karelian.to_csv("album_Karelian.csv")

In [11]:
speed = bands_long.query("genre == 'Speed'")["artiste"]
speed_songs = songs.query("artiste in @speed").copy()
speed_songs.head()

,artiste,album,annee,titre,paroles
36044,Blind Guardian,Lucifer's Heritage: Symphonies Of Doom,1985,Halloween,[3:22]\n\nYears ago I came back.\nThis'll be m...
36045,Blind Guardian,Lucifer's Heritage: Symphonies Of Doom,1985,Brian,"[2:41]\n\nWhen the sky is burning,\nsee the Mo..."
36046,Blind Guardian,Lucifer's Heritage: Symphonies Of Doom,1985,Dead Of The Night,[3:33]\n\nEvery night I hear thunder.\nThe sha...
36047,Blind Guardian,Lucifer's Heritage: Symphonies Of Doom,1985,Symphonies Of Doom,[4:08]\n\n [Hansi:]\n\nIn the dark night (they...
36048,Blind Guardian,Lucifer's Heritage: Symphonies Of Doom,1985,Lucifer's Heritage,[4:36]\n\n [Instrumental]


In [12]:
album_Lucifer_heritage = speed_songs.query('album == "Lucifer\'s Heritage: Symphonies Of Doom"')[["titre","paroles"]]
album_Lucifer_heritage.to_csv("album_Lucifer_heritage.csv")

In [13]:
doom_songs["genre"] = "doom"
doom_songs = doom_songs.sort_values(["genre", "titre"]).reset_index(drop=True)
speed_songs["genre"] = "speed"
songs = pd.concat([doom_songs, speed_songs], ignore_index=True)
songs = songs.sort_values(["genre", "titre"]).reset_index(drop=True)
doom_songs

,artiste,album,annee,titre,paroles,genre
0,Black Sabbath,Black Sabbath,1970,A Bit Of Finger / Sleeping Village,"Red sun rising in the sky\nSleeping village, c...",doom
1,Black Sabbath,Sabbath Bloody Sabbath,1973,A National Acrobat,I am the world that hides\nThe universal secre...,doom
2,Amorphis,Circle,2013,A New Day,"[Composition: Holopainen, Esa]\n\nThe Sun will...",doom
3,Amorphis,Halo,2022,A New Land,We wandered through strange lands\nTraveling e...,doom
4,Amorphis,Silent Waters,2007,A Servant,What was denied from poor\nI thought to be ric...,doom
...,...,...,...,...,...,...
604,Paradise Lost,Shades Of God,1992,Your Hand In Mine,Pay the deadly price\nLift the fallen prize\nT...,doom
605,Paradise Lost,Seals The Sense,1994,Your Hand In Mine (live),Pay the deadly price\nLift the fallen prize\nT...,doom
606,Paradise Lost,In Requiem,2007,Your Own Reality,"All that's lost never found, remains of all th...",doom
607,Black Sabbath,13,2013,Zeitgeist,Astral engines in reverse\nI'm falling through...,doom
